## Árvores de regressão - exercícios 02

Este exercício será uma continuação do anterior, mesma base, mesmas variáveis - vamos tentar buscar a 'melhor árvore'.


*Atenção - Utilizar a base de dados em anexo que é a mesma base que utilizamos na atividade anterior! A base Boston, assim como para a primeira atividade foi descontinuada e não deve ser utilizada*

In [ ]:
import pandas as pd

import seaborn as sns

from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
from sklearn.model_selection import train_test_split

### 1. Execute os passos do exercício anterior, até que você tenha uma árvore de regressão predizendo o valor do imóvel na base de treinamento.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error # Para possível uso posterior, se necessário
import numpy as np

print("--- Árvores de Regressão - Exercício 02: Preparação ---")

# 1. Carregar o dataset
try:
    df_california = pd.read_csv('housing.csv')
    print("Dataset 'housing.csv' carregado com sucesso!")
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

# 2. Tratar valores ausentes na coluna 'total_bedrooms'
# Preencher com a mediana, conforme decidido anteriormente.
median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
print(f"\nValores ausentes em 'total_bedrooms' preenchidos com a mediana: {median_bedrooms}")

# 3. Codificar a variável categórica 'ocean_proximity' usando One-Hot Encoding
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)
print("\nVariável 'ocean_proximity' codificada com One-Hot Encoding.")

# Definir as features (X) e a variável target (y)
X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

print(f"\nVariável Target (y) definida. Formato de y: {y.shape}")
print(f"Features (X) definidas. Formato de X: {X.shape}")

# 4. Separar os dados em conjuntos de treino e teste
# Usaremos 80% para treino e 20% para teste com random_state=42 para reprodutibilidade.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nDados separados em conjuntos de treino e teste:")
print(f"Shape de X_train (features de treino): {X_train.shape}")
print(f"Shape de y_train (target de treino): {y_train.shape}")
print(f"Shape de X_test (features de teste): {X_test.shape}")
print(f"Shape de y_test (target de teste): {y_test.shape}")

# 5. Treinar uma árvore de regressão na base de treinamento
# Usaremos max_depth=8 como ponto de partida, conforme o exercício anterior.
print("\nTreinando uma Árvore de Regressão (com max_depth=8) na base de treinamento...")
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42)
dt_reg_base.fit(X_train, y_train)
print("Árvore de Regressão base treinada com sucesso!")

# Opcional: fazer uma previsão na base de treinamento apenas para confirmar que o modelo funciona
y_pred_train_base = dt_reg_base.predict(X_train)
mse_train_base = mean_squared_error(y_train, y_pred_train_base)
print(f"MSE na base de TREINAMENTO (Árvore Base): {mse_train_base:,.2f}")

### 2.  Calcule o caminho indicado pelos CCP-alfas dessa árvore.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import numpy as np

# --- Re-carregar e pré-processar o DataFrame e treinar a árvore base ---
# (Este bloco garante que o código seja executável de forma independente e que dt_reg_base esteja disponível)
try:
    df_california = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)

X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar a árvore base (com max_depth=8, como no exercício anterior)
# Para a poda por complexidade de custo ser mais abrangente, idealmente
# a árvore inicial seria treinada sem restrição de profundidade (max_depth=None).
# No entanto, seguindo a sequência do exercício, utilizaremos a árvore já definida.
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42)
dt_reg_base.fit(X_train, y_train)
# --- Fim do bloco de preparação ---

print("--- Calculando o Caminho de Poda por Complexidade de Custo (CCP-Alphas) ---")

# Calcular o caminho de poda (ccp_alphas e impurezas correspondentes)
# A poda é baseada na redução da impureza (MSE neste caso para regressão)
path = dt_reg_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

print(f"Número total de ccp_alphas encontrados: {len(ccp_alphas)}")

print("\nPrimeiros 10 ccp_alphas gerados (representam os pontos de poda):")
print(ccp_alphas[:10])

print("\nÚltimos 10 ccp_alphas gerados:")
print(ccp_alphas[-10:])

### 3. Paca cada valor de alpha obtido no item 2, treine uma árvore com o respectivo alfa, e guarde essa árvore em uma lista.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import numpy as np

# --- Re-carregar e pré-processar o DataFrame e obter os ccp_alphas ---
# (Este bloco garante que o código seja executável de forma independente)
try:
    df_california = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)

X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar a árvore base (com max_depth=8, para obter os ccp_alphas dela)
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42)
dt_reg_base.fit(X_train, y_train)

# Obter os ccp_alphas do caminho de poda
path = dt_reg_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities
# --- Fim do bloco de preparação ---

print("--- Treinando Árvores para Cada Valor de CCP-Alpha ---")

# Lista para armazenar as árvores treinadas
clfs = [] # 'clfs' é uma abreviação comum para 'classifiers', mas aqui são 'regressors'

# Loop sobre cada valor de alpha
for ccp_alpha in ccp_alphas:
    # Cria uma nova árvore de regressão com o ccp_alpha atual
    # random_state é importante para reprodutibilidade se houver empates
    # max_depth=None aqui é importante para que a poda seja controlada APENAS pelo ccp_alpha
    # mas estamos usando a mesma base da árvore anterior que tinha max_depth=8.
    # O scikit-learn já considera o path dessa árvore específica.
    reg = DecisionTreeRegressor(random_state=42, ccp_alpha=ccp_alpha)
    reg.fit(X_train, y_train)
    clfs.append(reg)

print(f"\nTotal de árvores treinadas e guardadas na lista: {len(clfs)}")
print("A lista 'clfs' agora contém uma árvore para cada ccp_alpha do caminho de poda.")

### 4. Para cada árvore na lista, calcule o MSE da árvore.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Re-carregar e pré-processar o DataFrame e obter os ccp_alphas e treinar as árvores ---
# (Este bloco garante que o código seja executável de forma independente)
try:
    df_california = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)

X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar a árvore base para obter os ccp_alphas dela
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42)
dt_reg_base.fit(X_train, y_train)

# Obter os ccp_alphas
path = dt_reg_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Treinar as árvores para cada alpha (reproduzindo o passo anterior)
clfs = []
for ccp_alpha in ccp_alphas:
    reg = DecisionTreeRegressor(random_state=42, ccp_alpha=ccp_alpha)
    reg.fit(X_train, y_train)
    clfs.append(reg)
# --- Fim do bloco de preparação ---

print("--- Calculando MSE para cada Árvore na Lista (em Treino e Teste) ---")

# Listas para armazenar os MSEs
mse_train = []
mse_test = []

# Loop sobre cada árvore treinada na lista 'clfs'
for c in clfs:
    # Previsões na base de treinamento
    y_pred_train = c.predict(X_train)
    mse_train.append(mean_squared_error(y_train, y_pred_train))

    # Previsões na base de teste
    y_pred_test = c.predict(X_test)
    mse_test.append(mean_squared_error(y_test, y_pred_test))

print(f"\nNúmero de MSEs calculados para treino: {len(mse_train)}")
print(f"Número de MSEs calculados para teste: {len(mse_test)}")

print("\nOs MSEs foram calculados para cada árvore.")

### 5. Monte um gráfico do MSE pelo alpha, escolha um valor de alpha perto do ponto de mínimo do MSE

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# --- Re-carregar e pré-processar o DataFrame e obter os ccp_alphas e treinar as árvores e calcular MSEs ---
# (Este bloco garante que o código seja executável de forma independente)
try:
    df_california = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)

X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar a árvore base para obter os ccp_alphas dela
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42)
dt_reg_base.fit(X_train, y_train)

# Obter os ccp_alphas
path = dt_reg_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Treinar as árvores para cada alpha
clfs = []
for ccp_alpha in ccp_alphas:
    reg = DecisionTreeRegressor(random_state=42, ccp_alpha=ccp_alpha)
    reg.fit(X_train, y_train)
    clfs.append(reg)

# Calcular os MSEs
mse_train = []
mse_test = []
for c in clfs:
    y_pred_train = c.predict(X_train)
    mse_train.append(mean_squared_error(y_train, y_pred_train))
    y_pred_test = c.predict(X_test)
    mse_test.append(mean_squared_error(y_test, y_pred_test))
# --- Fim do bloco de preparação ---

print("--- Plotando MSE vs. CCP-Alpha ---")

# Criar o gráfico
plt.figure(figsize=(12, 6))
plt.plot(ccp_alphas, mse_train, marker='o', label='MSE (Treino)', drawstyle="steps-post")
plt.plot(ccp_alphas, mse_test, marker='o', label='MSE (Teste)', drawstyle="steps-post")
plt.xlabel("Alpha (ccp_alpha)")
plt.ylabel("Erro Quadrático Médio (MSE)")
plt.title("MSE vs. Alpha para Diferentes Árvores Podadas")
plt.legend()
plt.grid(True)
plt.xscale('log') # Usar escala logarítmica para o alpha, pois os valores podem variar muito
plt.show()

# --- Escolhendo um valor de Alpha Perto do Mínimo do MSE de Teste ---
# Encontrar o índice do menor MSE no conjunto de teste
optimal_alpha_idx = np.argmin(mse_test)
optimal_alpha = ccp_alphas[optimal_alpha_idx]
min_mse_test = mse_test[optimal_alpha

### 6. Calcule o R-quadrado dessa árvore encontrada no item acima

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score # Importar r2_score
import numpy as np
import matplotlib.pyplot as plt

# --- Re-carregar e pré-processar o DataFrame e obter os ccp_alphas e treinar as árvores e calcular MSEs ---
# (Este bloco garante que o código seja executável de forma independente)
try:
    df_california = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)

X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar a árvore base para obter os ccp_alphas dela
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42)
dt_reg_base.fit(X_train, y_train)

# Obter os ccp_alphas
path = dt_reg_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Treinar as árvores para cada alpha e calcular MSEs (para encontrar o optimal_alpha)
clfs = []
mse_test = []
for ccp_alpha in ccp_alphas:
    reg = DecisionTreeRegressor(random_state=42, ccp_alpha=ccp_alpha)
    reg.fit(X_train, y_train)
    clfs.append(reg)
    y_pred_test = reg.predict(X_test)
    mse_test.append(mean_squared_error(y_test, y_pred_test))

# Encontrar o ccp_alpha ótimo (do ponto de mínimo do MSE de teste)
optimal_alpha_idx = np.argmin(mse_test)
optimal_alpha = ccp_alphas[optimal_alpha_idx]
min_mse_test = mse_test[optimal_alpha_idx]

print(f"Alpha ótimo encontrado (baseado no menor MSE de teste): {optimal_alpha:.6f}")
print(f"Menor MSE de teste: {min_mse_test:,.2f}")
# --- Fim do bloco de preparação e identificação do alpha ótimo ---


print("\n--- Calculando R-quadrado para a 'Melhor Árvore' ---")

# 1. Treinar a árvore com o ccp_alpha ótimo
dt_reg_optimal = DecisionTreeRegressor(random_state=42, ccp_alpha=optimal_alpha)
dt_reg_optimal.fit(X_train, y_train)
print(f"Árvore treinada com o optimal_alpha = {optimal_alpha:.6f}")

# 2. Fazer previsões na base de TESTE com a árvore ótima
y_pred_optimal_test = dt_reg_optimal.predict(X_test)

# 3. Calcular o R-quadrado na base de TESTE
r2_optimal_test = r2_score(y_test, y_pred_optimal_test)
print(f"R-quadrado (R²) na base de TESTE para a árvore ótima: {r2_optimal_test:.4f}")

# Opcional: Calcular R-quadrado na base de TREINAMENTO para comparação
y_pred_optimal_train = dt_reg_optimal.predict(X_train)
r2_optimal_train = r2_score(y_train, y_pred_optimal_train)
print(f"R-quadrado (R²) na base de TREINAMENTO para a árvore ótima: {r2_optimal_train:.4f}")

print("\n--- Interpretação do R-quadrado ---")
print("O R-quadrado indica a proporção da variância na variável alvo (median_house_value) que é explicada pelo modelo.")
print("Um R² de 0.0 significa que o modelo não explica nenhuma variabilidade.")
print("Um R² de 1.0 significa que o modelo explica toda a variabilidade da variável alvo.")
print("O R² na base de TESTE é o mais importante, pois indica a capacidade de generalização do modelo para dados não vistos.")

### 7. Visualize esta árvore.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# --- Re-carregar e pré-processar o DataFrame e obter os ccp_alphas e treinar a árvore ótima ---
# (Este bloco garante que o código seja executável de forma independente)
try:
    df_california = pd.read_csv('housing.csv')
except FileNotFoundError:
    print("Erro: 'housing.csv' não encontrado. Certifique-se de que o arquivo está no diretório correto.")
    exit()

median_bedrooms = df_california['total_bedrooms'].median()
df_california['total_bedrooms'] = df_california['total_bedrooms'].fillna(median_bedrooms)
df_processed = pd.get_dummies(df_california, columns=['ocean_proximity'], drop_first=False)

X = df_processed.drop('median_house_value', axis=1)
y = df_processed['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar a árvore base (para obter os ccp_alphas dela)
dt_reg_base = DecisionTreeRegressor(max_depth=8, random_state=42) # Usamos max_depth=8 para o path
dt_reg_base.fit(X_train, y_train)

# Obter os ccp_alphas
path = dt_reg_base.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Treinar e avaliar árvores para encontrar o optimal_alpha (reprodução dos passos anteriores)
clfs = []
mse_test = []
for ccp_alpha in ccp_alphas:
    reg = DecisionTreeRegressor(random_state=42, ccp_alpha=ccp_alpha)
    reg.fit(X_train, y_train)
    clfs.append(reg)
    y_pred_test = reg.predict(X_test)
    mse_test.append(mean_squared_error(y_test, y_pred_test))

# Encontrar o ccp_alpha ótimo
optimal_alpha_idx = np.argmin(mse_test)
optimal_alpha = ccp_alphas[optimal_alpha_idx]

# Treinar a árvore final com o ccp_alpha ótimo
dt_reg_optimal = DecisionTreeRegressor(random_state=42, ccp_alpha=optimal_alpha)
dt_reg_optimal.fit(X_train, y_train)
# --- Fim do bloco de preparação e treino da árvore ótima ---

print("--- Visualizando a Árvore de Regressão Ótima ---")
print(f"Alpha (ccp_alpha) utilizado para esta árvore: {optimal_alpha:.6f}")
print(f"Profundidade final da árvore: {dt_reg_optimal.get_depth()}")
print(f"Número de nós folha da árvore: {dt_reg_optimal.get_n_leaves()}")


# Plotar a árvore ótima
plt.figure(figsize=(25, 15)) # Ajuste o tamanho da figura para melhor visualização
plot_tree(dt_reg_optimal,
          feature_names=X_train.columns.tolist(), # Nomes das variáveis preditoras
          filled=True, # Preenche os nós com cores
          rounded=True, # Cantos arredondados
          fontsize=6, # Tamanho da fonte dos textos nos nós
          precision=2) # Precisão dos valores numéricos (MSE, value)
plt.title(f'Árvore de Regressão Ótima (ccp_alpha={optimal_alpha:.6f})')
plt.show()

print("\n--- Interpretação da Árvore Ótima Visualizada ---")
print("Esta visualização mostra a estrutura final da árvore de regressão após ser podada com o